# Full Agriculture Fine-tune Pipeline (Single Notebook)

This notebook is self-contained and runs the full workflow:

1. install dependencies
2. prepare dataset
3. build RAG corpus
4. fine-tune SmolVLM
5. export merged inference model
6. zip and download artifacts


In [ ]:
import sys, subprocess

def run(cmd):
    print('>>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)

run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
run([sys.executable, '-m', 'pip', 'install',
     'torch>=2.3.0', 'transformers>=4.48.0', 'datasets>=2.20.0',
     'accelerate>=0.30.0', 'peft>=0.12.0', 'bitsandbytes>=0.43.0',
     'Pillow>=10.0.0', 'tqdm>=4.66.0'])

In [ ]:
from pathlib import Path
import os, json, random

# Force single GPU visibility in notebook runtimes (prevents cuda:0/cuda:1 mismatches).
if 'CUDA_VISIBLE_DEVICES' not in os.environ:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
from datasets import load_dataset

ROOT = Path('/kaggle/working')
if not ROOT.exists():
    ROOT = Path('/content/agri_pipeline')

DATA_DIR = ROOT / 'data' / 'agriculture'
IMG_DIR = DATA_DIR / 'images'
RAG_DIR = ROOT / 'data' / 'rag'
OUT_DIR = ROOT / 'outputs' / 'smolvlm-agri-ft'
EXPORT_DIR = ROOT / 'models' / 'SmolVLM-256M-Instruct-Agri'

for d in [DATA_DIR, IMG_DIR, RAG_DIR, OUT_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'
MAX_TRAIN = 1200
MAX_EVAL = 300
EPOCHS = 1
USE_LORA = True
HAS_CUDA = torch.cuda.is_available()
USE_4BIT = HAS_CUDA
DEVICE = 'cuda:0' if HAS_CUDA else 'cpu'

print('ROOT:', ROOT)
print('CUDA available:', HAS_CUDA)
print('Visible GPUs:', os.environ.get('CUDA_VISIBLE_DEVICES', '(not set)'))
print('Device:', DEVICE)
print('Using 4-bit:', USE_4BIT)

In [ ]:
LABEL_TO_GUIDANCE = {
    'healthy': 'The leaf appears healthy. Maintain balanced irrigation and continue monitoring.',
    'angular_leaf_spot': 'Likely angular leaf spot. Remove heavily infected tissue, reduce leaf wetness, improve airflow.',
    'bean_rust': 'Likely bean rust. Remove infected leaves, reduce humidity, and apply labeled treatment when needed.'
}
QUESTION_TEMPLATES = [
    'What is the likely issue in this crop leaf image and what should I do next?',
    'Analyze this plant image and suggest practical treatment steps.',
    'Identify the crop condition from this image and recommend immediate actions.'
]

ds = load_dataset('beans')
label_names = ds['train'].features['labels'].names

def to_record(image_rel_path, label_name):
    q = random.choice(QUESTION_TEMPLATES)
    g = LABEL_TO_GUIDANCE.get(label_name, 'Plant stress detected. Isolate affected plants and monitor closely.')
    a = f'Diagnosis: {label_name}. Recommended action: {g}'
    return {'image': image_rel_path, 'question': q, 'answer': a}

def export_split(split_data, split_name, limit, out_jsonl):
    count = 0
    with open(out_jsonl, 'w', encoding='utf-8') as f:
        for i, sample in enumerate(split_data):
            if count >= limit:
                break
            image = sample['image'].convert('RGB')
            label_name = label_names[int(sample['labels'])]
            name = f'{split_name}_{i:06d}.jpg'
            image.save(IMG_DIR / name, format='JPEG', quality=95)
            rec = to_record(f'images/{name}', label_name)
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            count += 1
    return count

train_count = export_split(ds['train'], 'train', MAX_TRAIN, DATA_DIR / 'train.jsonl')
eval_source = ds['validation'] if 'validation' in ds else ds['test']
eval_count = export_split(eval_source, 'eval', MAX_EVAL, DATA_DIR / 'eval.jsonl')
print({'train_records': train_count, 'eval_records': eval_count})

In [ ]:
seed_docs = [
  {'id':'ipm-core','title':'Integrated Pest Management baseline','source':'seed','text':'Scout regularly, use pest thresholds, protect beneficial insects, rotate modes of action, and keep treatment logs.'},
  {'id':'fungal-control','title':'Fungal disease control basics','source':'seed','text':'Reduce leaf wetness, improve airflow, remove infected debris, sanitize tools, and apply fungicides only when needed.'},
  {'id':'soil-health','title':'Soil health basics','source':'seed','text':'Use soil tests, maintain crop-appropriate pH, improve organic matter, and avoid over-irrigation.'},
  {'id':'starter-routine','title':'Beginner farming routine','source':'seed','text':'Follow weekly scouting, irrigation checks, disease checks, weed control, and record keeping.'}
]

corpus = []
for d in seed_docs:
    corpus.append(d)

for src_name, fpath, limit in [
    ('train', DATA_DIR / 'train.jsonl', 600),
    ('eval', DATA_DIR / 'eval.jsonl', 200),
]:
    if not fpath.exists():
        continue
    for i, line in enumerate(fpath.read_text(encoding='utf-8').splitlines()):
        if i >= limit:
            break
        row = json.loads(line)
        q = str(row.get('question', '')).strip()
        a = str(row.get('answer', '')).strip()
        if not q or not a:
            continue
        corpus.append({
            'id': f'{src_name}-{i}',
            'title': 'Agriculture diagnosis and action',
            'source': src_name,
            'text': f'Question: {q}\nAnswer: {a}'
        })

with open(RAG_DIR / 'corpus.jsonl', 'w', encoding='utf-8') as f:
    for row in corpus:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('RAG docs:', len(corpus))

In [ ]:
import importlib
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed

set_seed(42)
dataset = load_dataset('json', data_files={'train': str(DATA_DIR / 'train.jsonl'), 'validation': str(DATA_DIR / 'eval.jsonl')})
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

quant_config = None
if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

model_kwargs = {'trust_remote_code': True, 'low_cpu_mem_usage': True}
if quant_config is not None:
    model_kwargs['quantization_config'] = quant_config
    # Keep everything on one GPU to avoid cuda:0/cuda:1 tensor mismatches.
    model_kwargs['device_map'] = {'': 0}

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **model_kwargs)
if quant_config is None and DEVICE.startswith('cuda'):
    model = model.to(DEVICE)

if USE_LORA:
    peft_mod = importlib.import_module('peft')
    LoraConfig = getattr(peft_mod, 'LoraConfig')
    get_peft_model = getattr(peft_mod, 'get_peft_model')
    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj','k_proj','v_proj','o_proj','up_proj','down_proj','gate_proj']
    )
    model = get_peft_model(model, lora_cfg)
    if hasattr(model, 'print_trainable_parameters'):
        model.print_trainable_parameters()

In [ ]:
class Collator:
    def __init__(self, processor, data_root):
        self.processor = processor
        self.data_root = Path(data_root)
        self.system_prompt = 'You are an agriculture assistant. Provide practical and safe guidance.'
        self.image_token_id = None
        try:
            toks = self.processor.tokenizer.additional_special_tokens
            ids = self.processor.tokenizer.additional_special_tokens_ids
            if '<image>' in toks:
                self.image_token_id = ids[toks.index('<image>')]
        except Exception:
            pass

    def __call__(self, features):
        texts, images = [], []
        for feat in features:
            q = str(feat.get('question', '')).strip()
            a = str(feat.get('answer', '')).strip()
            img_rel = str(feat.get('image', '')).strip()
            img_path = Path(img_rel)
            if not img_path.is_absolute():
                img_path = self.data_root / img_path
            image = Image.open(img_path).convert('RGB')
            messages = [
                {'role': 'system', 'content': [{'type': 'text', 'text': self.system_prompt}]},
                {'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': q}]},
                {'role': 'assistant', 'content': [{'type': 'text', 'text': a}]},
            ]
            text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            texts.append(text)
            images.append(image)

        batch = self.processor(text=texts, images=images, return_tensors='pt', padding=True)
        labels = batch['input_ids'].clone()
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        if self.image_token_id is not None:
            labels[labels == self.image_token_id] = -100
        batch['labels'] = labels
        return batch

collator = Collator(processor, DATA_DIR)

training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=EPOCHS,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=0,
    weight_decay=0.01,
    optim='paged_adamw_8bit' if quant_config is not None else 'adamw_torch',
    eval_strategy='steps',
    eval_steps=50,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to='none',
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    data_collator=collator,
)

trainer.train()
trainer.save_model(str(OUT_DIR))
processor.save_pretrained(str(OUT_DIR))
print('Training output:', OUT_DIR)

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

if USE_LORA:
    peft_mod = importlib.import_module('peft')
    PeftModel = getattr(peft_mod, 'PeftModel')
    base_model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, trust_remote_code=True, low_cpu_mem_usage=True)
    merged = PeftModel.from_pretrained(base_model, str(OUT_DIR)).merge_and_unload()
    merged.save_pretrained(str(EXPORT_DIR), safe_serialization=True)
    proc_src = str(OUT_DIR) if (OUT_DIR / 'processor_config.json').exists() else MODEL_ID
    AutoProcessor.from_pretrained(proc_src, trust_remote_code=True).save_pretrained(str(EXPORT_DIR))
else:
    AutoModelForImageTextToText.from_pretrained(str(OUT_DIR), trust_remote_code=True, low_cpu_mem_usage=True).save_pretrained(str(EXPORT_DIR), safe_serialization=True)
    AutoProcessor.from_pretrained(str(OUT_DIR), trust_remote_code=True).save_pretrained(str(EXPORT_DIR))

print('Exported model:', EXPORT_DIR)

In [ ]:
from shutil import make_archive

model_zip = make_archive(str(ROOT / 'smolvlm_agri_model'), 'zip', root_dir=str(EXPORT_DIR))
rag_zip = make_archive(str(ROOT / 'rag_corpus'), 'zip', root_dir=str(RAG_DIR))
print('Model zip:', model_zip)
print('RAG zip:', rag_zip)

try:
    from google.colab import files
    files.download(model_zip)
    files.download(rag_zip)
except Exception as e:
    print('Download helper not available in this environment:', e)
    print('Files are located in', ROOT)